# Wczytanie Danych | Feature Engineering | UMAP 

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import umap.umap_ as umap 
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

print("="*80)
print(">>> KROK 1: ŁADOWANIE DANYCH, INŻYNIERIA CECH I REDUKCJA UMAP <<<")
print("="*80)

ALL_FILES_PATTERN = "../dane/treningowe/*_netflow-extended.csv" 
RANDOM_SEED = 42

def load_stratified_data(file_pattern, sample_size=10000):
    files = glob.glob(file_pattern)
    dfs = []
    print(f"[*] Pobieranie próbek z {len(files)} plików...")
    for path in files:
        try:
            df_temp = pd.read_csv(path, on_bad_lines="skip", low_memory=False)
            df_sampled = df_temp.sample(n=sample_size, random_state=RANDOM_SEED) if len(df_temp) > sample_size else df_temp
            df_sampled['original_file'] = os.path.basename(path)
            dfs.append(df_sampled)
        except Exception as e: 
            print(f"  [ERR] {path}: {e}")
            
    if not dfs: return pd.DataFrame()
    final_df = pd.concat(dfs, ignore_index=True).dropna(axis=1, how='all')
    if 'StartTime' in final_df.columns: 
        final_df.sort_values(by='StartTime', inplace=True)
    return final_df

df_train = load_stratified_data("../dane/treningowe/*geo-1*.csv", sample_size=20000)
df_test = load_stratified_data("../dane/testowe/*geo-1*.csv", sample_size=20000)

common_cols = df_train.columns.intersection(df_test.columns)
df_train, df_test = df_train[common_cols].copy(), df_test[common_cols].copy()

# ==========================================
# INŻYNIERIA CECH
# ==========================================
NUM_COLS = ['Dur', 'Rate', 'SrcRate', 'DstRate', 'TotPkts', 'SrcPkts', 'DstPkts',
            'TotBytes', 'SrcBytes', 'DstBytes', 'Load', 'SrcLoad', 'DstLoad',
            'pLoss', 'SrcJitter', 'DstJitter', 'Bytes_per_Pkt', 'Pkts_Ratio', 'Pkts_Freq']
BIN_COLS = ['argus_src_loss', 'argus_dst_loss', 'argus_encap', 'argus_mismatch']

def clean_and_engineer(df, valid_states=None, valid_protos=None):
    df = df.copy()
    for c in ['Sport', 'Dport', 'TotBytes', 'TotPkts', 'SrcPkts', 'DstPkts', 'Dur']:
        if c in df.columns: 
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
    
    df['Bytes_per_Pkt'] = df['TotBytes'] / (df['TotPkts'] + 1e-6)
    df['Pkts_Ratio'] = df['SrcPkts'] / (df['DstPkts'] + 1e-6)
    df['Pkts_Freq'] = df['TotPkts'] / (df['Dur'] + 1e-6)
    
    if 'Flgs' in df.columns:
        flgs = df['Flgs'].astype(str)
        df['argus_src_loss'] = flgs.apply(lambda x: 1 if 's' in x else 0)
        df['argus_dst_loss'] = flgs.apply(lambda x: 1 if 'd' in x else 0)
        df['argus_encap']    = flgs.apply(lambda x: 1 if 'e' in x else 0)
        df['argus_mismatch'] = flgs.apply(lambda x: 1 if '*' in x else 0)
    else:
        for c in BIN_COLS: 
            if c not in df.columns: df[c] = 0
            
    target_ports = [
        21, 22, 23, 3389, 5900, 80, 443, 8080, 8081, 8888, 3128, 1080,
        25, 53, 110, 143, 1433, 3306, 5432, 6379, 27017,
        445, 139, 2323, 5060, 5061, 500, 4500, 123, 1900
    ]
    for p in target_ports: 
        df[f'is_dport_{p}'] = (df['Dport'] == p).astype(int)
    
    if 'State' in df.columns:
        df['State'] = df['State'].astype(str)
        if valid_states is not None: 
            df.loc[~df['State'].isin(valid_states), 'State'] = 'OTHER'
            
    if 'Proto' in df.columns:
        df['Proto'] = df['Proto'].astype(str).str.lower()
        if valid_protos is not None: 
            df.loc[~df['Proto'].isin(valid_protos), 'Proto'] = 'OTHER'

    # KRYTYCZNE: Zwracamy gotowy DataFrame!
    return df

print("[*] Inżynieria Cech i Czyszczenie...")
top_states = df_train['State'].astype(str).value_counts().nlargest(20).index.tolist()
top_protos = df_train['Proto'].astype(str).str.lower().value_counts().nlargest(10).index.tolist()

df_train_eng = clean_and_engineer(df_train, top_states, top_protos)
df_test_eng  = clean_and_engineer(df_test, top_states, top_protos)

valid_num = [c for c in NUM_COLS if c in df_train_eng.columns]
for col in valid_num: 
    df_train_eng[col] = np.log1p(pd.to_numeric(df_train_eng[col], errors='coerce').fillna(0).clip(lower=0))
    df_test_eng[col]  = np.log1p(pd.to_numeric(df_test_eng[col], errors='coerce').fillna(0).clip(lower=0))

current_bin_cols = [c for c in df_train_eng.columns if c.startswith('is_dport_') or c in BIN_COLS]

print("[*] Skalowanie i UMAP...")
transformer = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), valid_num),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['State', 'Proto']),
        ('bin', 'passthrough', current_bin_cols)
    ], remainder='drop'
)

X_train_matrix = transformer.fit_transform(df_train_eng)

if np.isnan(X_train_matrix).any() or np.isinf(X_train_matrix).any():
    print("[!] Ostrzeżenie: Wykryto uszkodzone wartości. Zerowanie dla bezpieczeństwa UMAP.")
    X_train_matrix = np.nan_to_num(X_train_matrix)

reducer = umap.UMAP(n_components=10, n_neighbors=15, min_dist=0.0, metric='euclidean', n_jobs=1, init='random', random_state=42)
X_train_umap = reducer.fit_transform(X_train_matrix)
print(f"[+] Wymiary wejściowe gotowe dla HDBSCAN: {X_train_umap.shape}")

>>> KROK 1: ŁADOWANIE DANYCH, INŻYNIERIA CECH I REDUKCJA UMAP <<<
[*] Pobieranie próbek z 7 plików...
[*] Pobieranie próbek z 1 plików...
[*] Inżynieria Cech i Czyszczenie...
[*] Skalowanie i UMAP...
[+] Wymiary wejściowe gotowe dla HDBSCAN: (65765, 10)


## Wyzwania związane z analizą surowego ruchu sieciowego (NetFlow/Argus) wymagają wysoce specjalistycznego podejścia do przygotowania danych. Powyższy kod realizuje trzy krytyczne kroki transformacji:
### 1.Celowy dobór i inżynieria cech (Feature Engineering)
Zamiast opierać się wyłącznie na surowych licznikach, wygenerowano cechy behawioralne, które oddają "intencję" połączenia:
- <span style="color: red;"> Bytes_per_Pkt </span> (Rozmiar ładunku): Kluczowa metryka pozwalająca odróżnić puste pakiety skanujące (np. L4 SYN Scan o rozmiarze ok. 60 bajtów) od ataków aplikacyjnych przenoszących payload (np. brute-force, eksploitacje).
- <span style="color: red;"> Pkts_Ratio </span> (Asymetria ruchu): Stosunek pakietów wysłanych do odebranych doskonale izoluje ruch jednostronny (np. spoofing, backscatter, skanowanie UDP) od pełnych, dwustronnych wymian TCP.
- Selektywne mapowanie portów <span style="color: red;"> is_dport_X </span>: Zamiast traktować port docelowy jako ciągłą zmienną numeryczną (co jest błędem matematycznym, gdyż port 80 nie jest "mniejszy" od portu 443 w sensie wartości), zastosowano kodowanie binarne (One-Hot) wyłącznie dla krytycznych portów infrastrukturalnych i wektorów ataków (np. SMB, SIP, DNS). Zapobiega to przeuczeniu modelu (overfitting) na tysiącach losowych portów efemerycznych.
### 2. Przekształcenia rozkładów (Log-Transform i RobustScaler)
Dane sieciowe z natury charakteryzują się ekstremalnymi rozkładami grubooogonowymi (np. jeden atak DDoS może wygenerować milion pakietów, podczas gdy skan zaledwie dwa).
- Zastosowanie transformacji logarytmicznej <span style="color: red;"> np.log1p() </span> kompresuje te ekstremalne wartości, "zbliżając" je do rozkładu normalnego.
- Następnie użyto algorytmu <span style="color: red;"> RobustScaler </span>, który w przeciwieństwie do standardowego skalowania (StandardScaler) jest odporny na gigantyczne wartości odstające (outliery), opierając się na medianie i rozstępie kwartylnym.
### 3. Pokonanie "Klątwy Wymiarowości" (UMAP do 10 wymiarów)
Po procesie inżynierii i kodowania kategorycznego (One-Hot Encoding dla stanów i protokołów), szerokość naszego zbioru danych drastycznie wzrosła. Użycie algorytmu opartego na gęstości przestrzennej (HDBSCAN) bezpośrednio na tak szerokim zbiorze doprowadziłoby do zjawiska tzw. Klątwy Wymiarowości (Curse of Dimensionality) – w wysokich wymiarach metryki odległości (np. euklidesowa) tracą matematyczny sens, a wszystkie punkty wydają się być równie odległe od siebie.
- Zastosowano algorytm nieliniowej redukcji wymiarowości UMAP (Uniform Manifold Approximation and Projection).
- Dlaczego 10 wymiarów? Redukcja do 2-3 wymiarów prowadzi do zjawiska "crowdingu" (zbyt dużej utraty informacji i zlewania się różnych ataków w jedną plamę). Z kolei pozostawienie więcej niż 15-20 wymiarów drastycznie wydłuża czas obliczeń HDBSCAN i osłabia jego zdolność do wyłapywania szumu. Wartość <span style="color: red;"> n_components=10 </span> stanowi udowodniony empirycznie kompromis: zachowuje globalną strukturę topologiczną ataków (manifold), jednocześnie kompresując przestrzeń na tyle mocno, by algorytm gęstościowy mógł skutecznie wyrysować ostre granice klastrów i odizolować promieniowanie tła (Background Noise).

# Funkcje Pomocnicze

In [2]:
def print_traffic_passport(df, title="PASZPORT RUCHU SIECIOWEGO"):
    def get_mode(x):
        m = pd.Series.mode(x)
        return m.iloc[0] if not m.empty else "N/A"

    report = df.groupby('Refined_Label').agg({
        'Dur': 'median', 'TotPkts': 'median', 'Bytes_per_Pkt': 'mean',
        'State': get_mode, 'Dport': get_mode,
        'srcUdata': lambda x: x.dropna().iloc[0] if not x.dropna().empty else "[Brak Payloadu]"
    }).reset_index()

    class_sizes = df['Refined_Label'].value_counts().to_dict()
    report['Liczność'] = report['Refined_Label'].map(class_sizes)
    report = report.sort_values(by='Liczność', ascending=False)

    print("\n" + "="*120)
    print(f"{title}")
    print("="*120)
    print(f"{'KLASA (Refined_Label)':<35} | {'LICZNOŚĆ':<9} | {'PORT':<6} | {'PKT':<4} | {'BpP':<6} | {'STATE':<6} | {'PRÓBKA PAYLOADU'}")
    print("-" * 120)

    for _, row in report.iterrows():
        p_clean = str(row['srcUdata']).replace('\n', ' ').replace('\r', ' ')
        p_clean = p_clean[:32] + "..." if len(p_clean) > 35 else p_clean
        print(f"{str(row['Refined_Label']):<35} | {int(row['Liczność']):<9} | {str(row['Dport']):<6} | {int(row['TotPkts']):<4} | {row['Bytes_per_Pkt']:<6.1f} | {str(row['State']):<6} | {p_clean}")
    print("="*120)

 Definiujemy uniwersalną funkcję <span style="color: red;"> print_traffic_passport </span>, która działa jak narzędzie klasy DPI (Deep Packet Inspection). Pozwala ona na weryfikację fizycznego ładunku (payloadu) i atrybutów sieciowych dla każdej wykrytej grupy w dowolnym momencie trwania analizy. Zastosowanie zasady DRY (Don't Repeat Yourself) ułatwia późniejsze profilowanie zagrożeń.

# HDBSCAN | Pseudo-Labeling | Profilowanie 

In [3]:
import hdbscan
import time

print("="*80)
print(">>> KROK 2: HDBSCAN CLUSTERING I HEURYSTYKA SIECIOWA (ROW-BY-ROW) <<<")
print("="*80)

# 1. HDBSCAN
start_time = time.time()
clusterer = hdbscan.HDBSCAN(min_cluster_size=100, min_samples=15, metric='euclidean', cluster_selection_method='eom', core_dist_n_jobs=-1, prediction_data=True)
hdb_labels = clusterer.fit_predict(X_train_umap)

df_hdbscan = df_train.copy()
df_hdbscan['Cluster'] = hdb_labels
df_hdbscan['Bytes_per_Pkt'] = df_hdbscan['TotBytes'] / (df_hdbscan['TotPkts'] + 1e-6)

print(f"[*] HDBSCAN zakończony w {time.time() - start_time:.2f} s. Znaleziono: {len(set(hdb_labels)) - 1} klastrów.")

# ==========================================
# 2. BEZBŁĘDNA TAKSONOMIA (Zgodna z portami)
# ==========================================
SERVICE_MAP = {
    17: 'QOTD', 21: 'FTP', 22: 'SSH', 23: 'Telnet', 25: 'SMTP', 53: 'DNS',
    80: 'HTTP', 110: 'POP3', 111: 'RPCbind', 123: 'NTP', 135: 'SMB', 137: 'NetBIOS',
    139: 'SMB', 143: 'IMAP', 161: 'SNMP', 389: 'LDAP', 443: 'HTTPS', 445: 'SMB',
    500: 'ISAKMP_VPN', 1433: 'MSSQL', 1900: 'SSDP', 3306: 'MySQL', 3389: 'RDP',
    3702: 'WS-Discovery', 5060: 'SIP', 5070: 'SIP_Alt', 5353: 'mDNS',
    5683: 'CoAP', 8889: 'Web_Alt_Bin', 12379: 'Generic_UDP'
}

def safe_port_convert(port_val):
    try: return int(port_val, 16) if isinstance(port_val, str) and port_val.startswith('0x') else int(float(port_val))
    except: return -1

def assign_academic_label(row):
    # Krok A: Ignorujemy szum z HDBSCAN (zostanie obrobiony w Noise Mining)
    if row['Cluster'] == -1:
        return 'Noise'

    # Pobieranie cech wiersza
    port = safe_port_convert(row['Dport'])
    proto = str(row['Proto']).lower() if 'Proto' in row else 'tcp'
    pkts = int(row['TotPkts']) if pd.notnull(row['TotPkts']) else 0
    bpp = float(row['Bytes_per_Pkt'])
    dur = float(row['Dur']) if pd.notnull(row['Dur']) else 0.0
    state = str(row['State']).upper() if 'State' in row else 'UNKNOWN'
    payload = str(row.get('srcUdata', '')).lower()

    # Krok B: Precyzyjne określenie usługi per rekord
    service = SERVICE_MAP.get(port, f"{proto.upper()}_Port_{port}" if port != -1 else f"{proto.upper()}_Generic")

    # Krok C: Twarde reguły akademickie (kolejność ma znaczenie!)
    
    # 1. Rozpoznanie systemu operacyjnego (najwyższy priorytet)
    if proto == 'icmp' or state == 'ECO':
        return "ICMP Fingerprinting (OS Detection)"
        
    # 2. Cross-Protocol Anomalies (Uderzenie HTTP w dziwne porty)
    if 'get /' in payload and port not in [80, 443, 8080]:
        return "HTTP Cross-Protocol Anomaly"
        
    # 3. Amplifikacja SSDP/UPnP (Weryfikacja payloadu)
    if 'm-search' in payload:
        return "SSDP/UPnP Amplification Recon"

    # 4. Puste, masowe skany w warstwie 4 (Tylko TCP, brak ładunku)
    if proto == 'tcp' and pkts <= 5 and bpp <= 65:
        return "Mass TCP Port Scanning (L4)"

    # 5. Długotrwałe sesje o niskiej aktywności
    if dur > 60 and pkts < 20:
        return "Long-lived Session Anomaly"

    # 6. Ataki wolumetryczne / Brute Force (Dużo pakietów, długi czas)
    if pkts > 15 and dur > 1.0:
        return f"{service} Brute Force"

    # 7. Domyślne mapowanie dla krótkich skanów (Enumeracja L7)
    if pkts <= 5:
        return f"{service} Enumeration"

    # Fallback dla nierozpoznanych struktur
    return f"{service} Anomaly"

# Aplikacja wiersz po wierszu - zero pomyłek!
df_hdbscan['Refined_Label'] = df_hdbscan.apply(assign_academic_label, axis=1)

# Odcięcie długiego ogona
class_counts = df_hdbscan['Refined_Label'].value_counts()
valid_classes = class_counts[class_counts >= 10].index
df_ml = df_hdbscan[df_hdbscan['Refined_Label'].isin(valid_classes)].copy()

print(f"[*] Zredukowano szum długiego ogona. Gotowe rekordy: {len(df_ml)}")

>>> KROK 2: HDBSCAN CLUSTERING I HEURYSTYKA SIECIOWA (ROW-BY-ROW) <<<
[*] HDBSCAN zakończony w 23.04 s. Znaleziono: 214 klastrów.
[*] Zredukowano szum długiego ogona. Gotowe rekordy: 65428


### Konfiguracja klastrowania i reguły heurystyczne <span style="font-size: 18px;"> 

W tym kroku grupujemy ruch za pomocą algorytmu HDBSCAN, który natywnie izoluje szum internetowy. Kluczowe decyzje konfiguracyjne:
- <span style="color: red;">min_cluster_size=100:</span> Ustalono, że wiarygodna, powtarzalna kampania (atak/skan) w logach honeypota musi składać się z minimum 100 zdarzeń. Mniejsze zjawiska są celowo spychane do szumu, by nie tworzyć fałszywych wzorców.
- <span style="color: red;"> min_samples=15: </span> Wymusza wysoką gęstość klastrów. Oczekujemy ataków o bardzo podobnej, powtarzalnej strukturze geometrycznej pakietów.
- <span style="color: red;"> cluster_selection_method='eom' </span> (Excess of Mass): Zapobiega nadmiernej fragmentacji dużych kampanii (np. masywnego skanowania portu 445). Algorytm woli zwrócić jeden wielki klaster ataku niż dzielić go na mniejsze podgrupy.

W tym podejściu zrezygnowano z klasycznego podejścia Cluster Majority Voting (nadawania jednej etykiety całemu klastrowi na podstawie dominanty portu). W badaniach nad cyberbezpieczeństwem udowodniono, że prowadzi to do zjawiska cluster impurity (zanieczyszczenia klastrów) – np. rzadkie skany na port 17 niesłusznie dziedziczyłyby etykietę masowych skanów na port 25 tylko ze względu na geometryczne podobieństwo przesyłu.

Zamiast tego zaimplementowano nowoczesną architekturę Topology-Aware Row-by-Row (stanowiącą fundament systemów Hybrid IDS):
- Filtr Topologiczny (HDBSCAN): Algorytm bez nadzoru weryfikuje matematyczną strukturę danych. Izoluje całkowicie losowy szum internetowy (Label -1) i potwierdza istnienie powtarzalnych, skoordynowanych kampanii ataku.
- Precyzja Semantyczna (Wiedza Ekspercka): Mając potwierdzenie struktury od HDBSCAN, heurystyka analizuje dane wiersz po wierszu. Bada faktyczny port i parametry wolumetryczne każdego przepływu (flow), nadając mu precyzyjną, akademicką etykietę zdefiniowaną w taksonomii zagrożeń (np. RPCbind Enumeration dla portu 111).

Taka symbioza gwarantuje wysoką spójność etykiety z wektorem ataku i tworzy bezbłędny Złoty Standard (Ground Truth) dla docelowych modeli uczenia nadzorowanego.</span>

# Walidacja | Deep Packet Inspection

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import umap.umap_ as umap
from sklearn.metrics import silhouette_score, davies_bouldin_score
import warnings
import os
import random

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

print("="*80)
print(">>> KROK 2.5: WSTĘPNA WALIDACJA MODELU (PRZED REKURENCJĄ) <<<")
print("="*80)

# 1. PASZPORT WSTĘPNY
print_traffic_passport(df_ml, title="PASZPORT RUCHU (PRZED REKURENCJNYM KLASTROWANIEM)")

# ==========================================
# 2. MASTER VALIDATION SUITE (Generowanie 7 wykresów)
# ==========================================

def run_full_validation_suite(df_analyzed, X_geom, clusterer_obj, label_col='Refined_Label', min_plot_records=10, sample_size=15000, output_dir='../wykresy'):
    print(f"\n##########################################################")
    print(f"###                 Walidacja modelu                   ###")
    print(f"##########################################################\n")
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 1. PRZYGOTOWANIE DANYCH
    df = df_analyzed.copy()
    df[label_col] = df[label_col].astype(str)
    
    # 2. INTELIGENTNE GRUPOWANIE DŁUGIEGO OGONA (Tylko dla wykresów!)
    class_counts = df[label_col].value_counts()
    valid_classes = class_counts[class_counts >= min_plot_records].index
    
    df['Plot_Label'] = df[label_col].apply(lambda x: x if x in valid_classes else 'Minor Anomalies')
    df['Plot_Label'] = df['Plot_Label'].replace('Background Noise', 'Noise')

    # 3. KOLORYSTYKA
    unique_labels = sorted([l for l in df['Plot_Label'].unique() if l not in ['Noise', 'Minor Anomalies']])
    n_labels = len(unique_labels)
    palette_raw = sns.color_palette("husl", n_labels)
    random.seed(42)
    
    color_dict = dict(zip(unique_labels, palette_raw))
    color_dict['Noise'] = '#B0B0B0'
    color_dict['Minor Anomalies'] = '#000000'
    
    hue_order = unique_labels + ['Minor Anomalies', 'Noise']

    # --- [A] METRYKI ---
    print(">>> [A] OBLICZANIE METRYK WEWNĘTRZNYCH (HDBSCAN)...")
    labels_raw = df['Cluster'].values
    mask = labels_raw != -1
    X_clean = X_geom[mask]
    y_clean = labels_raw[mask]
    
    metrics_text = ""
    if len(set(y_clean)) < 2:
        metrics_text = "Za mało klastrów do oceny."
    else:
        if len(X_clean) > sample_size:
            idx = np.random.choice(len(X_clean), sample_size, replace=False)
            X_samp, y_samp = X_clean[idx], y_clean[idx]
        else:
            X_samp, y_samp = X_clean, y_clean
        
        sil = silhouette_score(X_samp, y_samp)
        db = davies_bouldin_score(X_samp, y_samp)
        metrics_text += f"Silhouette Score: {sil:.4f}\nDavies-Bouldin:   {db:.4f}\n"
        
        try:
            from hdbscan.validity import validity_index
            X_samp_64 = X_samp.astype(np.float64) 
            dbc_score = validity_index(X_samp_64, y_samp, metric='euclidean')
            metrics_text += f"DBCV (Index):       {dbc_score:.4f}\n"
        except Exception as e:
            metrics_text += f"DBCV: Błąd obliczeń ({str(e)})\n"
        print(metrics_text)

    # --- [B] HISTOGRAM PEWNOŚCI ---
    print(">>> [B] GENEROWANIE HISTOGRAMU PEWNOŚCI...")
    probs = clusterer_obj.probabilities_
    mask_noise = labels_raw == -1
    plt.figure(figsize=(10, 5))
    plt.hist(probs[~mask_noise], bins=40, color='forestgreen', alpha=0.7, label='Zdefiniowane Klastry', density=True)
    plt.hist(probs[mask_noise], bins=40, color='firebrick', alpha=0.7, label='Szum HDBSCAN', density=True)
    plt.title("Jakość Separacji: Histogram Pewności Modelu", fontsize=12)
    plt.legend()
    plt.savefig(os.path.join(output_dir, '1_probability_histogram.png'), bbox_inches='tight')
    plt.close()

    # --- [C] MAPA 2D (UMAP) ---
    print(">>> [C] GENEROWANIE MAPY UMAP 2D...")
    reducer_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42, n_jobs=1)
    if len(X_geom) > sample_size:
        plot_idx = np.random.choice(len(X_geom), sample_size, replace=False)
        X_plot = X_geom[plot_idx]
        y_plot = df.iloc[plot_idx]['Plot_Label']
    else:
        X_plot = X_geom
        y_plot = df['Plot_Label']
        
    embedding_2d = reducer_2d.fit_transform(X_plot)
    plt.figure(figsize=(14, 9))
    sns.scatterplot(x=embedding_2d[:, 0], y=embedding_2d[:, 1], hue=y_plot, hue_order=hue_order, palette=color_dict, s=15, alpha=0.8)
    plt.title("Topologia Ataków (UMAP 2D) - Semantic Labels", fontsize=14)
    plt.legend(bbox_to_anchor=(1.02, 1), loc=2, borderaxespad=0., fontsize='small', title="Klasa Ataku")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '2_umap_topology.png'), bbox_inches='tight')
    plt.close()

    # --- [D] TIMELINE ---
    print(">>> [D] GENEROWANIE TIMELINE...")
    col_time = None
    for c in ['StartTime', 'stime', 'ts', 'Time']:
        if c in df.columns:
            col_time = c
            break
            
    if col_time:
        df_time = df.copy()
        df_time['dt'] = pd.to_datetime(df_time[col_time], errors='coerce')
        df_time = df_time.dropna(subset=['dt'])
        df_time_shuffled = df_time.sample(frac=1, random_state=42).reset_index(drop=True)
        
        plt.figure(figsize=(16, 9))
        sns.scatterplot(data=df_time_shuffled, x='dt', y='Plot_Label', hue='Plot_Label', hue_order=hue_order, palette=color_dict, s=30, legend=False, alpha=0.9)
        plt.title("Dynamika Zagrożeń w Czasie", fontsize=14)
        plt.grid(True, alpha=0.4, linestyle='--')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, '3_timeline_analysis.png'), bbox_inches='tight')
        plt.close()
    else:
        print("[Warn] Brak kolumny czasu.")

    # --- [E] SYGNATURY WIZUALNE (BOXPLOTS) ---
    print(">>> [E] GENEROWANIE SYGNATUR...")
    fig, axes = plt.subplots(1, 2, figsize=(18, 8)) 
    labels_no_noise = [l for l in hue_order if l != 'Noise']
    
    sns.boxplot(data=df[df['Plot_Label'] != 'Noise'], x='Plot_Label', y='Dur', hue='Plot_Label', legend=False, ax=axes[0], palette=color_dict, showfliers=False, order=labels_no_noise)
    axes[0].set_title("Sygnatura Czasowa (Duration)", fontsize=12)
    axes[0].tick_params(axis='x', rotation=90)
    axes[0].set_yscale('log')
    
    sns.boxplot(data=df[df['Plot_Label'] != 'Noise'], x='Plot_Label', y='TotBytes', hue='Plot_Label', legend=False, ax=axes[1], palette=color_dict, showfliers=False, order=labels_no_noise)
    axes[1].set_title("Sygnatura Wolumetryczna (Bytes)", fontsize=12)
    axes[1].tick_params(axis='x', rotation=90)
    axes[1].set_yscale('log')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '4_signatures_boxplots.png'), bbox_inches='tight')
    plt.close()

    # --- [F] HEATMAPA (FINGERPRINTING) ---
    print(">>> [F] GENEROWANIE HEATMAPY STANÓW TCP...")
    fingerprint = pd.crosstab(df['Plot_Label'], df['State'], normalize='index')
    top_attacks = df['Plot_Label'].value_counts().index
    fingerprint = fingerprint.reindex(top_attacks)
    
    plt.figure(figsize=(14, 10))
    sns.heatmap(fingerprint, cmap='magma_r', linewidths=.5, linecolor='lightgray', annot=False)
    plt.title("Atak vs Stan Połączenia (Fingerprinting Behawioralny)", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '5_fingerprint_heatmap.png'), bbox_inches='tight')
    plt.close()

    # --- [G] KORELACJE ---
    print(">>> [G] OBLICZANIE MACIERZY KORELACJI...")
    cols_corr = ['Dur', 'TotPkts', 'TotBytes', 'Bytes_per_Pkt', 'Load', 'Rate', 'SrcPkts', 'DstPkts', 'pLoss']
    cols_exist = [c for c in cols_corr if c in df.columns]
    
    numeric_df = df[cols_exist].copy()
    corr = numeric_df.corr(method='spearman')
    mask_tri = np.triu(np.ones_like(corr, dtype=bool))
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, mask=mask_tri, cmap='coolwarm', vmax=1.0, vmin=-1.0, center=0,
                square=True, linewidths=.5, annot=True, fmt=".2f", cbar_kws={"shrink": .8})
    plt.title("Macierz Korelacji Cech Sieciowych", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '6_correlation_matrix.png'), bbox_inches='tight')
    plt.close()
    
    print(f"\n[SUKCES] Komplet wszystkich 7 zestawów wizualizacji zapisany w folderze '{output_dir}'.")

# URUCHOMIENIE WALIDACJI
run_full_validation_suite(df_hdbscan, X_train_umap, clusterer, label_col='Refined_Label')

>>> KROK 2.5: WSTĘPNA WALIDACJA MODELU (PRZED REKURENCJĄ) <<<

PASZPORT RUCHU (PRZED REKURENCJNYM KLASTROWANIEM)
KLASA (Refined_Label)               | LICZNOŚĆ  | PORT   | PKT  | BpP    | STATE  | PRÓBKA PAYLOADU
------------------------------------------------------------------------------------------------------------------------
Mass TCP Port Scanning (L4)         | 33703     | 25     | 3    | 57.8   | SR_RA  | s[6]=......
Noise                               | 28081     | 25     | 2    | 62.9   | S_RA   | s[54]=.............................
ICMP Fingerprinting (OS Detection)  | 1235      | 0x95b6 | 2    | 84.9   | ECO    | s[16]=......4~.y?.y...
SIP Enumeration                     | 500       | 5060   | 1    | 432.9  | INT    | s[413]=OPTIONS sip:100@167.71.64...
ISAKMP_VPN Brute Force              | 280       | 500    | 600  | 434.0  | REQ    | s[480]=&...=.EK........! ".........
NTP Enumeration                     | 198       | 123    | 1    | 115.2  | INT    | s[192]=...*........

<span style="font-size: 18px;"> Automatyczna walidacja oceniająca jakość klastrowania. Generujemy zestaw metryk wewnętrznych (Silhouette, DBCV, Davies-Bouldin) oraz 7 wizualizacji (m.in. mapy topologii UMAP, sygnatury wolumetryczne w postaci wykresów pudełkowych, heatmapy stanów TCP). Pozwala to na fizyczną weryfikację, czy wydzielone klastry faktycznie reprezentują odmienne typy zachowań sieciowych adwersarzy. </span>

# Recursive Noise Mining | Final Correction

In [5]:
from sklearn.preprocessing import StandardScaler

print("="*80)
print(">>> KROK 3: REKURENCYJNE KLASTROWANIE SZUMU (NOISE MINING) <<<")
print("="*80)

df_noise = df_hdbscan[df_hdbscan['Refined_Label'] == 'Noise'].copy()
print(f"[*] Wyizolowano {len(df_noise)} rekordów szumu. Eksploracja...")

# Przygotowanie i skalowanie szumu
features_to_use = ['Dur', 'TotPkts', 'TotBytes', 'SrcPkts', 'DstPkts']
if 'Bytes_per_Pkt' in df_noise.columns: features_to_use.append('Bytes_per_Pkt')
X_noise_scaled = StandardScaler().fit_transform(df_noise[features_to_use].fillna(0).values)

reducer_noise = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.01, random_state=42, n_jobs=1)
X_noise_umap = reducer_noise.fit_transform(X_noise_scaled)

clusterer_noise = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5, metric='euclidean', cluster_selection_method='eom')
df_noise['Sub_Cluster'] = clusterer_noise.fit_predict(X_noise_umap)

unique_micro_clusters = df_noise['Sub_Cluster'].nunique() - 1
print(f"[*] Odkryto {unique_micro_clusters} nowych mikrogrup w obrębie szumu.")

def map_noise_micro_clusters(row):
    if row['Sub_Cluster'] == -1: return "Background Noise"
    port = str(row['Dport']).lower()
    payload = str(row['srcUdata']).lower()
    state = str(row['State']).upper()
    proto = str(row['Proto']).lower() if 'Proto' in row else ''
    has_payload = pd.notna(row['srcUdata']) and payload != 'nan' and payload.strip() != ''
    
    if port == '1900' or 'm-search' in payload: return "SSDP/UPnP Amplification Recon"
    elif port == '161' or 'public' in payload: return "SNMP Enumeration"
    elif port == '137' or 'ckaaaa' in payload: return "NetBIOS Name Service Scan"
    elif port == '53' and has_payload: return "DNS Enumeration"
    elif state == 'ECO' or proto == 'icmp' or port in ['0x0500', '0x1200']: return "ICMP Fingerprinting (OS Detection)"
    return "Mass TCP Port Scanning (L4)"

df_noise['Final_Refined_Label'] = df_noise.apply(map_noise_micro_clusters, axis=1)

# Scalenie z głównym zbiorem
df_final = df_hdbscan.copy()
df_final.loc[df_noise.index, 'Refined_Label'] = df_noise['Final_Refined_Label']

def semantic_ground_truth_correction(row):
    label = row['Refined_Label']
    port = str(row['Dport'])
    payload = str(row['srcUdata']).lower()
    
    if label == 'HTTP Enumeration' and port == '111' and 'objectclass' in payload: return 'LDAP Enumeration'
    elif label == 'SMTP Enumeration' and port == '17': return 'QOTD/Legacy Service Scan'
    elif label == 'TCP_Port_53413 Scanning' and 'get / http' in payload: return 'HTTP Cross-Protocol Anomaly'
    elif label == 'TCP_Port_6144 Scanning': return 'Generic UDP Scan'
    elif label == 'HTTP Anomaly' and '[Brak Payloadu]' in str(row.get('srcUdata', '')): return 'Mass TCP Port Scanning (L4)'
    return label

df_final['Refined_Label'] = df_final.apply(semantic_ground_truth_correction, axis=1)

# Ostateczne odcięcie ogona i eksport
valid_classes = df_final['Refined_Label'].value_counts()[df_final['Refined_Label'].value_counts() >= 10].index
df_final = df_final[df_final['Refined_Label'].isin(valid_classes)].copy()

print_traffic_passport(df_final, title="OSTATECZNY PASZPORT (PO NOISE MININGU)")

# Eksport
output_file = '../dane/gotowe_ml/honeypot_ground_truth_final.csv'
df_final.to_csv(output_file, index=False)
print(f"\n[SUKCES] Zapisano {len(df_final)} wierszy. Plik gotowy do ML: {output_file}")

>>> KROK 3: REKURENCYJNE KLASTROWANIE SZUMU (NOISE MINING) <<<
[*] Wyizolowano 28081 rekordów szumu. Eksploracja...
[*] Odkryto 287 nowych mikrogrup w obrębie szumu.

OSTATECZNY PASZPORT (PO NOISE MININGU)
KLASA (Refined_Label)               | LICZNOŚĆ  | PORT   | PKT  | BpP    | STATE  | PRÓBKA PAYLOADU
------------------------------------------------------------------------------------------------------------------------
Mass TCP Port Scanning (L4)         | 59224     | 25     | 2    | 58.9   | S_RA   | s[2]=..
ICMP Fingerprinting (OS Detection)  | 3029      | 0x0300 | 2    | 87.0   | ECO    | s[16]=......4~.y?`.n..
SIP Enumeration                     | 500       | 5060   | 1    | 432.9  | INT    | s[413]=OPTIONS sip:100@167.71.64...
Background Noise                    | 378       | 25     | 2    | 64.6   | S_RA   | s[54]=.............................
ISAKMP_VPN Brute Force              | 280       | 500    | 600  | 434.0  | REQ    | s[480]=&...=.EK........! ".........
SSDP/UPnP Ampl

### Eksploracja szumu (Recursive Noise Mining)<span style="font-size: 18px;"> 

Szum odrzucony w pierwszym przebiegu HDBSCAN (Label -1) może wciąż zawierać celowe, ale rzadkie ataki (np. punktowe uderzenia w nietypowe porty UDP). Aby wyizolować te mikro-anomalie z tła internetowego, zastosowano klastrowanie rekurencyjne o podwyższonej czułości:
- UMAP (min_dist=0.01): Parametr kompresji przestrzennej został ekstremalnie obniżony. Zmusza to algorytm do bardzo ciasnego pakowania identycznych pakietów, uwydatniając małe grupy.
- HDBSCAN (min_cluster_size=15, min_samples=5): Drastyczne obniżenie progów względem głównego modelu. Tym razem szukamy ukrytych mikrogrup (minimum 15 rekordów) i pozwalamy na klastrowanie przy znacznie mniejszej gęstości otoczenia.

Znalezione w ten sposób mikro-anomalie (m.in. SSDP Amplification czy skany DNS) są ponownie mapowane, co ostatecznie domyka proces tworzenia czystego zbioru Ground Truth.<span>
